In [103]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer

In [ ]:
def get_labels(data):
    stata_reader = pd.read_stata(data, iterator=True)
    labels_dict = stata_reader.variable_labels()
    data_dictionary = pd.DataFrame(columns=["Variable", "Label"], data=list(labels_dict.items()))
    return data_dictionary

In [109]:
aggregate = pd.read_stata("../Data/Kendall/aggregate.dta") # precint-level
aggregate_labels = get_labels("../Data/Kendall/aggregate.dta")

individual = pd.read_stata("../Data/Kendall/individual.dta") # individual-levels survey
individual_labels = get_labels("../Data/Kendall/individual.dta")




## Aggregate Data Exploration

In [ ]:
# Observations (95)
aggregate.shape

In [ ]:
# Variabel labels
aggregate_labels

In [ ]:
# Data types
aggregate.info()

In [ ]:
# Descriptive Statistis
aggregate.describe()

### Randomization balance: aggregate
Test balance of pre-treatment characteristics across treatment group (Table A1 Replication).

In [ ]:
models = []
dep_vars = ["mun11_tenrolled", "giovi", "fiorentina", "saione", "giotto"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm"]
for var in dep_vars:
    reg = smf.ols(formula=f"{var} ~ +{'+'.join(regressors)}", data=aggregate,).fit(cov_type="cluster",
                                                                                  cov_kwds={"groups" : aggregate["building"]})
    models.append(reg)

In [ ]:
table_A1 = Stargazer(models)

labels = ["Valence by Phone", "Valence by mail", "Ideology by phone", "Ideology by mail", "Double by phone", "Double by mail"]
table_A1.rename_covariates(dict(zip(regressors, labels)))
table_A1.custom_columns(["Eligible Voters", "First neighborhood","Second neighborhood", "Third neighborhood", "Fourth neighborhood" ])
table_A1.show_model_numbers(False)
table_A1.title("Ex-Ante Balancing Tests at the Precinct Level")
table_A1.covariate_order(regressors)
table_A1.show_adj_r2 = False
table_A1.show_confidence_intervals(False)
table_A1.show_degrees_of_freedom(False)
table_A1.show_f_statistic = False
table_A1.show_residual_std_err = False

### Individual Data Exploration

In [ ]:
individual_labels

In [ ]:
individual.shape

In [ ]:
individual.dtypes

In [ ]:
individual.describe()

In [ ]:
individual.vote_f.sum() # 746 people voted

### Randomization balance: individual

Test balance of pre-treatment characteristics across trreatment groups at individual levels (Table A2 replication).

In [ ]:
individual["male"] = np.where(individual["s1_gender"] == "female",0,1)
dep_vars = ["male", "over65", "college", "outlf", "white", "other", "centerleft", "house", "press", "tv"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm", "C(s1_date1)"]
models = []

for var in dep_vars:
    reg = smf.ols(formula=f"{var} ~ {'+'.join(regressors)}", data=individual).fit(cov_type="cluster",
                                                                                  cov_kwds={"groups" : individual["section"]})
    models.append(reg)
models

In [85]:
table_A2 = Stargazer(models)

table_A2.rename_covariates(dict(zip(regressors, labels)))
table_A2.custom_columns(["Male", "Over65", "College Graduate", "Out of labor force", "White collar", "Other occupation", "Center-left", "Home owner", "Read the press", "Watch TV"])
table_A2.show_model_numbers(False)
table_A2.title("Ex-Ante Balancing Tests at the Individual Level")
table_A2.covariate_order(regressors[:6])
table_A2.show_adj_r2 = False
table_A2.show_confidence_intervals(False)
table_A2.show_degrees_of_freedom(False)
table_A2.show_f_statistic = False
table_A2.show_residual_std_err = False
table_A2

## Table 3: Aggregate Effects

In [102]:
dep_vars = ["mun11_turnout", "mun11_pfanfani_cand", "mun11_pfanfani_parties"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm"]
models = []

for var in dep_vars:
    reg = smf.ols(formula=f"{var} ~ {'+'.join(regressors)}", data=aggregate).fit(cov_type="cluster",
                                                                                 cov_kwds={"groups" : aggregate["building"]})
    models.append(reg)
models


In [89]:
table3 = Stargazer(models)

table3.rename_covariates(dict(zip(regressors, labels)))
table3.custom_columns(["Turnout", "Incumber share", "Incumbent parties"])
table3.show_model_numbers(False)
table3.title("Reduced-Form Individual Estimates, All Groups")
table3.covariate_order(regressors[:6])
table3.show_adj_r2 = False
table3.show_confidence_intervals(False)
table3.show_degrees_of_freedom(False)
table3.show_f_statistic = False
table3.show_residual_std_err = False
table3

## Table 4: Individual-level evidence

In [98]:
dep_vars = ["turnout", "vote_f", "vote_party_f"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm", "C(s1_date1)"]
models = []

for var in dep_vars:
    mod = smf.ols(formula=f"{var} ~ {'+'.join(regressors)}", data=individual)
    reg = mod.fit(cov_type="cluster", cov_kwds={"groups" : individual.loc[mod.data.row_labels, "section"]})
    models.append(reg)
                                                                                
models

In [100]:
table3 = Stargazer(models)

table3.rename_covariates(dict(zip(regressors, labels)))
table3.custom_columns(["Turnout", "Incumber share", "Incumbent parties"])
table3.show_model_numbers(False)
table3.title("Reduced-Form Aggregate Estimates, All Groups")
table3.covariate_order(regressors[:6])
table3.show_adj_r2 = False
table3.show_confidence_intervals(False)
table3.show_degrees_of_freedom(False)
table3.show_f_statistic = False
table3.show_residual_std_err = False
table3